In [1]:
# # ============================================================
# # CELL 1: Imports and global config
# # ============================================================

# from pathlib import Path
# import gc
# import os
# import time
# import copy

# import numpy as np
# import pandas as pd
# import soundfile as sf
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from tqdm.auto import tqdm

# from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration
# from comet import download_model, load_from_checkpoint

# from numcodecs import Quantize, Zstd


# PROJECT_ROOT = Path("/home/paperspace/projects/iwslt2026-compression")
# OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "experiment_results"
# EXPERIMENT_NAME = "custom_codec_global_q2_zstd_eval_only_en_zh"
# EXP_DIR = OUTPUT_ROOT / EXPERIMENT_NAME
# EXP_DIR.mkdir(parents=True, exist_ok=True)

# MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
# DEVICE = "cuda:0"
# PROMPT_TEXT = "Translate this English speech into Chinese."
# MAX_NEW_TOKENS = 64
# COMET_BATCH_SIZE = 16

# EVAL_MANIFEST = PROJECT_ROOT / "data" / "manifests" / "acl6060_eval_en_zh.csv"

# print("project root exists:", PROJECT_ROOT.exists())
# print("cuda available:", torch.cuda.is_available())
# if torch.cuda.is_available():
#     print("gpu:", torch.cuda.get_device_name(0))


# # ============================================================
# # CELL 2: Load eval fold, processor, COMET
# # ============================================================

# eval_df = pd.read_csv(EVAL_MANIFEST).copy()
# eval_df["audio_path"] = eval_df["audio_path"].apply(lambda p: str((PROJECT_ROOT / p).resolve()))

# processor = AutoProcessor.from_pretrained(MODEL_ID)

# comet_path = download_model("Unbabel/wmt22-comet-da")
# comet_model = load_from_checkpoint(comet_path)

# print("eval rows:", len(eval_df))
# print(eval_df.head(2))


# # ============================================================
# # CELL 3: Custom codec-based compression
# # ============================================================

# class CodecPipeline:
#     def __init__(self, filters, compressor=None, encoded_dtype="f2"):
#         self.filters = filters
#         self.compressor = compressor
#         self.encoded_dtype = np.dtype(encoded_dtype)

#     def encode(self, arr):
#         x = np.asarray(arr, dtype=np.float32)

#         for f in self.filters:
#             x = f.encode(x)

#         x = np.asarray(x, dtype=self.encoded_dtype)
#         filtered_shape = x.shape
#         filtered_bytes = x.tobytes(order="C")

#         if self.compressor is not None:
#             filtered_bytes = self.compressor.encode(filtered_bytes)

#         return {
#             "payload": filtered_bytes,
#             "filtered_shape": filtered_shape,
#             "filtered_dtype": self.encoded_dtype.str,
#         }

#     def decode(self, encoded_obj, original_shape, original_dtype=np.float32):
#         payload = encoded_obj["payload"]
#         filtered_shape = tuple(encoded_obj["filtered_shape"])
#         filtered_dtype = np.dtype(encoded_obj["filtered_dtype"])

#         if self.compressor is not None:
#             payload = self.compressor.decode(payload)

#         x = np.frombuffer(payload, dtype=filtered_dtype).copy().reshape(filtered_shape)

#         for f in reversed(self.filters):
#             x = f.decode(x)

#         x = np.asarray(x, dtype=original_dtype).reshape(original_shape)
#         return x


# class CompressedLinear(nn.Module):
#     def __init__(self, linear_layer, pipeline, description):
#         super().__init__()

#         weight = linear_layer.weight.detach().cpu().numpy().astype(np.float32)

#         self.weight_shape = weight.shape
#         self.in_features = linear_layer.in_features
#         self.out_features = linear_layer.out_features
#         self.pipeline = pipeline
#         self.description = description

#         self.encoded_weight = self.pipeline.encode(weight)

#         if linear_layer.bias is not None:
#             self.bias = nn.Parameter(linear_layer.bias.detach().clone(), requires_grad=False)
#         else:
#             self.bias = None

#         self._cached_weight = None
#         self._cached_device = None
#         self._cached_dtype = None

#     def _get_cached_weight(self, device, dtype):
#         if (
#             self._cached_weight is None
#             or self._cached_device != device
#             or self._cached_dtype != dtype
#         ):
#             decoded = self.pipeline.decode(
#                 self.encoded_weight,
#                 original_shape=self.weight_shape,
#                 original_dtype=np.float32,
#             )
#             self._cached_weight = torch.tensor(decoded, device=device, dtype=dtype)
#             self._cached_device = device
#             self._cached_dtype = dtype

#         return self._cached_weight

#     def forward(self, x):
#         weight = self._get_cached_weight(x.device, x.dtype)
#         return F.linear(x, weight, self.bias)


# def get_module_by_name(model, module_name):
#     module = model
#     for part in module_name.split("."):
#         module = getattr(module, part)
#     return module


# def set_module_by_name(model, module_name, new_module):
#     parts = module_name.split(".")
#     parent = model
#     for part in parts[:-1]:
#         parent = getattr(parent, part)
#     setattr(parent, parts[-1], new_module)


# def get_linear_layer_names(model):
#     names = []
#     for name, module in model.named_modules():
#         if isinstance(module, nn.Linear):
#             names.append(name)
#     return names


# def estimate_linear_storage_bytes(layer):
#     if isinstance(layer, nn.Linear):
#         total = layer.weight.detach().cpu().numpy().nbytes
#         if layer.bias is not None:
#             total += layer.bias.detach().cpu().numpy().nbytes
#         return total
#     if isinstance(layer, CompressedLinear):
#         total = len(layer.encoded_weight["payload"])
#         if layer.bias is not None:
#             total += layer.bias.detach().cpu().numpy().nbytes
#         return total
#     return 0


# def compress_model_global_inplace(model, pipeline, description):
#     layer_names = get_linear_layer_names(model)

#     for layer_name in tqdm(layer_names, desc="compress_linear_layers"):
#         layer = get_module_by_name(model, layer_name)
#         compressed_layer = CompressedLinear(layer, pipeline, description)
#         set_module_by_name(model, layer_name, compressed_layer)

#     return model


# def build_storage_report(model_before, model_after):
#     original_layers = dict(model_before.named_modules())
#     compressed_layers = dict(model_after.named_modules())

#     rows = []
#     total_original = 0
#     total_compressed = 0

#     for name, orig_layer in original_layers.items():
#         comp_layer = compressed_layers.get(name, None)

#         if isinstance(orig_layer, nn.Linear):
#             orig_bytes = estimate_linear_storage_bytes(orig_layer)
#             comp_bytes = estimate_linear_storage_bytes(comp_layer)

#             total_original += orig_bytes
#             total_compressed += comp_bytes

#             rows.append(
#                 {
#                     "layer_name": name,
#                     "original_bytes": orig_bytes,
#                     "compressed_bytes": comp_bytes,
#                     "compression_ratio": orig_bytes / comp_bytes if comp_bytes > 0 else np.nan,
#                     "method": getattr(comp_layer, "description", "N/A"),
#                 }
#             )

#     report_df = pd.DataFrame(rows)
#     totals = {
#         "original_total_bytes": total_original,
#         "compressed_total_bytes": total_compressed,
#         "total_ratio": total_original / total_compressed if total_compressed > 0 else np.nan,
#     }
#     return report_df, totals


# # ============================================================
# # CELL 4: Inference and evaluation helpers
# # ============================================================

# def generate_translation(model, processor, audio_path, prompt_text=PROMPT_TEXT, max_new_tokens=MAX_NEW_TOKENS):
#     audio, sr = sf.read(str(audio_path))

#     if audio.ndim > 1:
#         audio = audio.mean(axis=1)

#     conversation = [
#         {
#             "role": "user",
#             "content": [
#                 {"type": "audio", "audio_url": str(audio_path)},
#                 {"type": "text", "text": prompt_text},
#             ],
#         }
#     ]

#     text = processor.apply_chat_template(
#         conversation,
#         add_generation_prompt=True,
#         tokenize=False,
#     )

#     inputs = processor(
#         text=text,
#         audios=[audio],
#         sampling_rate=sr,
#         return_tensors="pt",
#     )

#     inputs = {
#         k: v.to(DEVICE) if hasattr(v, "to") else v
#         for k, v in inputs.items()
#     }

#     with torch.no_grad():
#         generated_ids = model.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             do_sample=False,
#             num_beams=1,
#             use_cache=True,
#         )

#     generated_ids = generated_ids[:, inputs["input_ids"].size(1):]

#     pred = processor.batch_decode(
#         generated_ids,
#         skip_special_tokens=True,
#         clean_up_tokenization_spaces=False,
#     )[0].strip()

#     return pred


# def run_inference(model, df_input, run_name):
#     preds = []
#     start = time.time()

#     for _, row in tqdm(df_input.iterrows(), total=len(df_input), desc=run_name):
#         pred = generate_translation(model, processor, Path(row["audio_path"]))
#         preds.append(pred)

#     elapsed = time.time() - start

#     out_df = df_input.copy()
#     out_df["prediction"] = preds
#     return out_df, elapsed


# def compute_comet(pred_df, comet_model, batch_size=COMET_BATCH_SIZE):
#     comet_data = [
#         {"src": r["source_en"], "mt": r["prediction"], "ref": r["target_zh"]}
#         for _, r in pred_df.iterrows()
#     ]

#     comet_out = comet_model.predict(comet_data, batch_size=batch_size, gpus=1)
#     comet_score = comet_out.system_score if hasattr(comet_out, "system_score") else comet_out[1]
#     return comet_score


# def cleanup_model(model):
#     del model
#     gc.collect()
#     torch.cuda.empty_cache()


# # ============================================================
# # CELL 5: Load baseline model on CPU, compress in place, build storage report
# # ============================================================

# baseline_cpu = Qwen2AudioForConditionalGeneration.from_pretrained(
#     MODEL_ID,
#     torch_dtype=torch.float16,
#     device_map="cpu",
# )
# baseline_cpu.eval()

# model_before = copy.deepcopy(baseline_cpu)

# global_pipeline = CodecPipeline(
#     filters=[Quantize(digits=2, dtype="f4", astype="f2")],
#     compressor=Zstd(level=3),
#     encoded_dtype="f2",
# )
# global_desc = 'Global custom codec: Quantize(digits=2, astype="f2") + Zstd'

# quant_model = compress_model_global_inplace(
#     baseline_cpu,
#     pipeline=global_pipeline,
#     description=global_desc,
# )

# storage_report_df, storage_totals = build_storage_report(model_before, quant_model)

# storage_report_path = EXP_DIR / "custom_codec_storage_report.csv"
# storage_report_df.to_csv(storage_report_path, index=False)

# print("storage report saved to:", storage_report_path)
# print(storage_totals)

# del model_before
# gc.collect()


# # ============================================================
# # CELL 6: Move compressed model to GPU and run eval inference
# # ============================================================

# quant_model = quant_model.to(DEVICE)
# quant_model.eval()

# quant_df, quant_time = run_inference(
#     quant_model,
#     eval_df,
#     run_name="custom_codec_global_eval",
# )

# preds_path = EXP_DIR / "custom_codec_global_eval_preds.csv"
# quant_df.to_csv(preds_path, index=False, encoding="utf-8")

# quant_comet = compute_comet(quant_df, comet_model)

# print("predictions saved to:", preds_path)
# print("eval seconds:", round(quant_time, 2))
# print("eval COMET:", quant_comet)


# # ============================================================
# # CELL 7: Save summary spreadsheets
# # ============================================================

# summary_df = pd.DataFrame(
#     [
#         {
#             "experiment_name": EXPERIMENT_NAME,
#             "run": "custom_codec_global_eval",
#             "split": "eval",
#             "rows": len(quant_df),
#             "seconds": quant_time,
#             "comet": quant_comet,
#             "original_total_bytes": storage_totals["original_total_bytes"],
#             "compressed_total_bytes": storage_totals["compressed_total_bytes"],
#             "compression_ratio": storage_totals["total_ratio"],
#             "compression_method": global_desc,
#             "model_id": MODEL_ID,
#             "max_new_tokens": MAX_NEW_TOKENS,
#         }
#     ]
# )

# summary_path = EXP_DIR / "custom_codec_global_eval_summary.csv"
# summary_df.to_csv(summary_path, index=False)

# print("summary saved to:", summary_path)
# print(summary_df)


# # ============================================================
# # CELL 8: Cleanup
# # ============================================================

# cleanup_model(quant_model)

In [2]:
# ============================================================
# CELL 1: Imports and global config
# ============================================================

from pathlib import Path
import gc
import os
import time
import copy

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration
from comet import download_model, load_from_checkpoint

from numcodecs import Quantize, Zstd


PROJECT_ROOT = Path("/home/paperspace/projects/iwslt2026-compression")
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "experiment_results"
EXPERIMENT_NAME = "custom_codec_global_q2_zstd_eval_only_en_zh"
EXP_DIR = OUTPUT_ROOT / EXPERIMENT_NAME
EXP_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
DEVICE = "cuda:0"
PROMPT_TEXT = "Translate this English speech into Chinese."
MAX_NEW_TOKENS = 64
COMET_BATCH_SIZE = 16

EVAL_MANIFEST = PROJECT_ROOT / "data" / "manifests" / "acl6060_eval_en_zh.csv"

print("project root exists:", PROJECT_ROOT.exists())
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


# ============================================================
# CELL 2: Load eval fold, processor, COMET
# ============================================================

eval_df = pd.read_csv(EVAL_MANIFEST).copy()
eval_df["audio_path"] = eval_df["audio_path"].apply(lambda p: str((PROJECT_ROOT / p).resolve()))

processor = AutoProcessor.from_pretrained(MODEL_ID)

comet_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(comet_path)

print("eval rows:", len(eval_df))
print(eval_df.head(2))


# ============================================================
# CELL 3: Custom codec-based compression
# ============================================================

class CodecPipeline:
    def __init__(self, filters, compressor=None, encoded_dtype="f2"):
        self.filters = filters
        self.compressor = compressor
        self.encoded_dtype = np.dtype(encoded_dtype)

    def encode(self, arr):
        x = np.asarray(arr, dtype=np.float32)

        for f in self.filters:
            x = f.encode(x)

        x = np.asarray(x, dtype=self.encoded_dtype)
        filtered_shape = x.shape
        filtered_bytes = x.tobytes(order="C")

        if self.compressor is not None:
            filtered_bytes = self.compressor.encode(filtered_bytes)

        return {
            "payload": filtered_bytes,
            "filtered_shape": filtered_shape,
            "filtered_dtype": self.encoded_dtype.str,
        }

    def decode(self, encoded_obj, original_shape, original_dtype=np.float32):
        payload = encoded_obj["payload"]
        filtered_shape = tuple(encoded_obj["filtered_shape"])
        filtered_dtype = np.dtype(encoded_obj["filtered_dtype"])

        if self.compressor is not None:
            payload = self.compressor.decode(payload)

        x = np.frombuffer(payload, dtype=filtered_dtype).copy().reshape(filtered_shape)

        for f in reversed(self.filters):
            x = f.decode(x)

        x = np.asarray(x, dtype=original_dtype).reshape(original_shape)
        return x


class CompressedLinear(nn.Module):
    def __init__(self, linear_layer, pipeline, description):
        super().__init__()

        weight = linear_layer.weight.detach().cpu().numpy().astype(np.float32)

        self.weight_shape = weight.shape
        self.in_features = linear_layer.in_features
        self.out_features = linear_layer.out_features
        self.pipeline = pipeline
        self.description = description

        self.encoded_weight = self.pipeline.encode(weight)

        if linear_layer.bias is not None:
            self.bias = nn.Parameter(linear_layer.bias.detach().clone(), requires_grad=False)
        else:
            self.bias = None

        self._cached_weight = None
        self._cached_device = None
        self._cached_dtype = None

    def _get_cached_weight(self, device, dtype):
        if (
            self._cached_weight is None
            or self._cached_device != device
            or self._cached_dtype != dtype
        ):
            decoded = self.pipeline.decode(
                self.encoded_weight,
                original_shape=self.weight_shape,
                original_dtype=np.float32,
            )
            self._cached_weight = torch.tensor(decoded, device=device, dtype=dtype)
            self._cached_device = device
            self._cached_dtype = dtype

        return self._cached_weight

    def forward(self, x):
        weight = self._get_cached_weight(x.device, x.dtype)
        return F.linear(x, weight, self.bias)


def get_module_by_name(model, module_name):
    module = model
    for part in module_name.split("."):
        module = getattr(module, part)
    return module


def set_module_by_name(model, module_name, new_module):
    parts = module_name.split(".")
    parent = model
    for part in parts[:-1]:
        parent = getattr(parent, part)
    setattr(parent, parts[-1], new_module)


def get_linear_layer_names(model):
    names = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            names.append(name)
    return names


def estimate_linear_storage_bytes(layer):
    if isinstance(layer, nn.Linear):
        total = layer.weight.detach().cpu().numpy().nbytes
        if layer.bias is not None:
            total += layer.bias.detach().cpu().numpy().nbytes
        return total
    if isinstance(layer, CompressedLinear):
        total = len(layer.encoded_weight["payload"])
        if layer.bias is not None:
            total += layer.bias.detach().cpu().numpy().nbytes
        return total
    return 0


def estimate_model_disk_size_bytes(model_dir):
    model_dir = Path(model_dir)
    total = 0
    if model_dir.exists():
        for fp in model_dir.rglob("*"):
            if fp.is_file():
                total += fp.stat().st_size
    return total


def compress_model_global_inplace(model, pipeline, description):
    layer_names = get_linear_layer_names(model)

    for layer_name in tqdm(layer_names, desc="compress_linear_layers"):
        layer = get_module_by_name(model, layer_name)
        compressed_layer = CompressedLinear(layer, pipeline, description)
        set_module_by_name(model, layer_name, compressed_layer)

    return model


def build_storage_report(model_before, model_after):
    original_layers = dict(model_before.named_modules())
    compressed_layers = dict(model_after.named_modules())

    rows = []
    total_original = 0
    total_compressed = 0

    for name, orig_layer in original_layers.items():
        comp_layer = compressed_layers.get(name, None)

        if isinstance(orig_layer, nn.Linear):
            orig_bytes = estimate_linear_storage_bytes(orig_layer)
            comp_bytes = estimate_linear_storage_bytes(comp_layer)

            total_original += orig_bytes
            total_compressed += comp_bytes

            rows.append(
                {
                    "layer_name": name,
                    "original_linear_bytes_est": orig_bytes,
                    "compressed_linear_bytes_est": comp_bytes,
                    "compression_ratio_est": orig_bytes / comp_bytes if comp_bytes > 0 else np.nan,
                    "size_scope": "estimated_linear_layers",
                    "method": getattr(comp_layer, "description", "N/A"),
                }
            )

    report_df = pd.DataFrame(rows)
    totals = {
        "original_total_linear_bytes_est": total_original,
        "compressed_total_linear_bytes_est": total_compressed,
        "total_ratio_est": total_original / total_compressed if total_compressed > 0 else np.nan,
        "size_scope": "estimated_linear_layers",
    }
    return report_df, totals


# ============================================================
# CELL 4: Inference and evaluation helpers
# ============================================================

def generate_translation(model, processor, audio_path, prompt_text=PROMPT_TEXT, max_new_tokens=MAX_NEW_TOKENS):
    audio, sr = sf.read(str(audio_path))

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": str(audio_path)},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]

    text = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=False,
    )

    inputs = processor(
        text=text,
        audios=[audio],
        sampling_rate=sr,
        return_tensors="pt",
    )

    inputs = {
        k: v.to(DEVICE) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
        )

    generated_ids = generated_ids[:, inputs["input_ids"].size(1):]

    pred = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    return pred


def run_inference(model, df_input, run_name):
    preds = []
    start = time.time()

    for _, row in tqdm(df_input.iterrows(), total=len(df_input), desc=run_name):
        pred = generate_translation(model, processor, Path(row["audio_path"]))
        preds.append(pred)

    elapsed = time.time() - start

    out_df = df_input.copy()
    out_df["prediction"] = preds
    return out_df, elapsed


def compute_comet(pred_df, comet_model, batch_size=COMET_BATCH_SIZE):
    comet_data = [
        {"src": r["source_en"], "mt": r["prediction"], "ref": r["target_zh"]}
        for _, r in pred_df.iterrows()
    ]

    comet_out = comet_model.predict(comet_data, batch_size=batch_size, gpus=1)
    comet_score = comet_out.system_score if hasattr(comet_out, "system_score") else comet_out[1]
    return comet_score


def cleanup_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()


# ============================================================
# CELL 5: Load baseline model on CPU, compress in place, build estimated linear-storage report
# ============================================================

baseline_cpu = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",
)
baseline_cpu.eval()

model_before = copy.deepcopy(baseline_cpu)

global_pipeline = CodecPipeline(
    filters=[Quantize(digits=2, dtype="f4", astype="f2")],
    compressor=Zstd(level=3),
    encoded_dtype="f2",
)
global_desc = 'Global custom codec: Quantize(digits=2, astype="f2") + Zstd'

quant_model = compress_model_global_inplace(
    baseline_cpu,
    pipeline=global_pipeline,
    description=global_desc,
)

storage_report_df, storage_totals = build_storage_report(model_before, quant_model)
# NOTE: these are estimated linear-layer storage bytes, not full model size on disk.

storage_report_path = EXP_DIR / "custom_codec_estimated_linear_storage_report.csv"
storage_report_df.to_csv(storage_report_path, index=False)

print("storage report saved to:", storage_report_path)
print(storage_totals)

del model_before
gc.collect()


# ============================================================
# CELL 6: Move compressed model to GPU and run eval inference
# ============================================================

quant_model = quant_model.to(DEVICE)
quant_model.eval()

quant_df, quant_time = run_inference(
    quant_model,
    eval_df,
    run_name="custom_codec_global_eval",
)

preds_path = EXP_DIR / "custom_codec_global_eval_preds.csv"
quant_df.to_csv(preds_path, index=False, encoding="utf-8")

quant_comet = compute_comet(quant_df, comet_model)

print("predictions saved to:", preds_path)
print("eval seconds:", round(quant_time, 2))
print("eval COMET:", quant_comet)


# ============================================================
# CELL 7: Save summary spreadsheets
# ============================================================

summary_df = pd.DataFrame(
    [
        {
            "experiment_name": EXPERIMENT_NAME,
            "run": "custom_codec_global_eval",
            "split": "eval",
            "rows": len(quant_df),
            "seconds": quant_time,
            "comet": quant_comet,
            "original_total_linear_bytes_est": storage_totals["original_total_linear_bytes_est"],
            "compressed_total_linear_bytes_est": storage_totals["compressed_total_linear_bytes_est"],
            "compression_ratio_est": storage_totals["total_ratio_est"],
            "size_scope": storage_totals["size_scope"],
            "compression_method": global_desc,
            "model_id": MODEL_ID,
            "max_new_tokens": MAX_NEW_TOKENS,
            "model_size_on_disk_gb": np.nan,
            "compression_ratio_disk": np.nan,
        }
    ]
)

summary_path = EXP_DIR / "custom_codec_global_eval_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("summary saved to:", summary_path)
print(summary_df)


# ============================================================
# CELL 8: Cleanup
# ============================================================

cleanup_model(quant_model)

project root exists: True
cuda available: True
gpu: NVIDIA RTX A6000


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/home/paperspace/miniconda3/envs/iwslt-2026/lib/python3.11/site-packages/lightning_fabric/utilities/cloud_io.py:52: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torc

eval rows: 416
       id split                                         audio_path  \
0  eval_1  eval  /home/paperspace/projects/iwslt2026-compressio...   
1  eval_2  eval  /home/paperspace/projects/iwslt2026-compressio...   

                                           source_en  \
0  Hi everyone. Today I'm going to present our re...   
1  I'm Allan from ByteDance AI Lab, and this is a...   

                                           target_zh  
0    大家好。今天我将介绍我们的研究工作：《学习演绎推理：作为复杂关系提取的数学文字问题解决方法》。  
1  我是ByteDance 人工智能实验室的Allan，以下是我与德克萨斯大学奥斯汀分校的Jie...  


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

compress_linear_layers:   0%|          | 0/418 [00:00<?, ?it/s]

storage report saved to: /home/paperspace/projects/iwslt2026-compression/outputs/experiment_results/custom_codec_global_q2_zstd_eval_only_en_zh/custom_codec_estimated_linear_storage_report.csv
{'original_total_linear_bytes_est': 15500451840, 'compressed_total_linear_bytes_est': 3895499044, 'total_ratio_est': 3.9790670373477313, 'size_scope': 'estimated_linear_layers'}


custom_codec_global_eval:   0%|          | 0/416 [00:00<?, ?it/s]

/home/paperspace/miniconda3/envs/iwslt-2026/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/paperspace/miniconda3/envs/iwslt-2026/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.5` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/paperspace/miniconda3/envs/iwslt-2026/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
We

predictions saved to: /home/paperspace/projects/iwslt2026-compression/outputs/experiment_results/custom_codec_global_q2_zstd_eval_only_en_zh/custom_codec_global_eval_preds.csv
eval seconds: 347.88
eval COMET: 0.7410914550463741
summary saved to: /home/paperspace/projects/iwslt2026-compression/outputs/experiment_results/custom_codec_global_q2_zstd_eval_only_en_zh/custom_codec_global_eval_summary.csv
                               experiment_name                       run  \
0  custom_codec_global_q2_zstd_eval_only_en_zh  custom_codec_global_eval   

  split  rows    seconds     comet  original_total_linear_bytes_est  \
0  eval   416  347.88201  0.741091                      15500451840   

   compressed_total_linear_bytes_est  compression_ratio_est  \
0                         3895499044               3.979067   

                size_scope                                 compression_method  \
0  estimated_linear_layers  Global custom codec: Quantize(digits=2, astype...   

            